# 🤖 Belajar Machine Learning dari Nol
## Tutorial Lengkap: Dari Data hingga Model di HuggingFace

---

**Untuk siapa tutorial ini?**
- Mahasiswa yang baru pertama kali belajar AI/ML
- Peneliti tanpa latar belakang IT
- Siapa saja yang ingin tahu cara kerja model AI

**Apa yang akan kamu pelajari?**

| No | Langkah | Analogi Sederhana |
|----|---------|-----------------|
| 1 | Membuat Dataset | Menyiapkan bahan masakan |
| 2 | Upload Dataset ke HuggingFace | Menyimpan bahan di gudang publik |
| 3 | Membuat Model | Menulis resep masakan |
| 4 | Melatih Model | Belajar memasak berulang kali |
| 5 | Fine-tuning BERT | Chef berpengalaman belajar menu baru |
| 6 | Evaluasi & Validasi | Mencicipi dan menilai masakan |
| 7 | Upload Model ke HuggingFace | Mempublikasikan resep untuk umum |

---

**Tema Simulasi:** Prediksi apakah sebuah ulasan film **positif** atau **negatif**

*(Tema sederhana, mudah dipahami semua orang)*

> 📌 **Cara pakai:** Buka Google Colab → Upload file ini → Jalankan setiap cell dari atas ke bawah
> 
> 🔗 Google Colab: [colab.research.google.com](https://colab.research.google.com)

---
## ⚙️ PERSIAPAN — Install Library

Library = alat bantu yang sudah dibuat orang lain agar kita tidak perlu buat dari nol.

Jalankan cell ini **sekali saja** di awal.

In [ ]:
# Install semua yang dibutuhkan
# Tanda ! di depan artinya perintah ke sistem, bukan Python
!pip install -q torch transformers datasets scikit-learn pandas numpy huggingface_hub

print('✅ Semua library berhasil diinstall!')
print('Library yang kita gunakan:')
print('  - torch       : framework deep learning utama (PyTorch)')
print('  - transformers: library untuk BERT dan model modern')
print('  - datasets    : library untuk HuggingFace Dataset')
print('  - sklearn     : tools evaluasi model (F1, accuracy, dll)')
print('  - pandas      : manipulasi data tabular')
print('  - numpy       : komputasi matematika')

In [ ]:
# ============================================================
# IMPORT SEMUA LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    roc_auc_score, confusion_matrix,
    classification_report
)
from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)
from datasets import Dataset as HFDataset, DatasetDict
import warnings
warnings.filterwarnings('ignore')

# Atur seed agar hasil selalu sama setiap kali dijalankan
# (penting untuk reproducibility - standar penelitian ilmiah)
torch.manual_seed(42)
np.random.seed(42)

# Deteksi apakah ada GPU (lebih cepat dari CPU)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('✅ Import berhasil!')
print(f'   PyTorch versi  : {torch.__version__}')
print(f'   Perangkat pakai: {DEVICE.upper()}')
print()
print('💡 Jika pakai GPU di Colab: Runtime → Change runtime type → T4 GPU')

---
## 📊 LANGKAH 1: Membuat Dataset

### Apa itu Dataset?
Dataset adalah kumpulan data yang menjadi bahan latihan model AI.
Seperti kumpulan contoh soal untuk belajar ujian.

### Yang kita buat:
- **Input (X)** : ulasan film dalam bentuk teks
- **Label (y)** : 1 = positif, 0 = negatif

Model akan belajar dari contoh-contoh ini untuk mengenali sentimen.

---

In [ ]:
# ============================================================
# MEMBUAT DATASET ULASAN FILM
# ============================================================

# Data ulasan film (kita buat manual untuk contoh)
# Dalam penelitian nyata, ini bisa ribuan atau jutaan data

ulasan_positif = [
    "Film yang sangat bagus, saya sangat terhibur",
    "Ceritanya menarik dan akting pemainnya luar biasa",
    "Recommended banget, wajib ditonton semua orang",
    "Alur cerita yang mengalir dan ending yang memuaskan",
    "Efek visualnya memukau dan musiknya sangat pas",
    "Saya menonton dua kali karena sangat bagus",
    "Film terbaik yang pernah saya tonton tahun ini",
    "Sutradara berhasil menyampaikan pesan dengan sangat baik",
    "Akting para pemain sangat natural dan menyentuh hati",
    "Jalan cerita yang tidak terduga membuat penonton penasaran",
    "Sinematografi yang indah dan dialog yang berkesan",
    "Film keluarga yang bisa dinikmati semua usia",
    "Sangat menghibur dan penuh dengan pesan moral yang baik",
    "Karakter yang kuat dan pengembangan cerita yang solid",
    "Saya tertawa dan menangis, film yang sangat emosional",
]

ulasan_negatif = [
    "Film yang sangat membosankan, saya hampir tertidur",
    "Cerita tidak masuk akal dan akting yang buruk",
    "Pembuangan waktu dan uang, tidak ada yang menarik",
    "Alur cerita yang membingungkan dan ending yang mengecewakan",
    "Efek visualnya murahan dan dialog yang kaku",
    "Baru setengah film saya sudah mau keluar bioskop",
    "Film terburuk yang pernah saya tonton",
    "Cerita yang klise dan tidak ada yang baru",
    "Akting pemain utama sangat kaku dan tidak meyakinkan",
    "Banyak adegan yang tidak perlu dan membuang waktu",
    "Kualitas produksi sangat buruk dan amatir",
    "Tidak sesuai ekspektasi sama sekali, sangat mengecewakan",
    "Iklan filmnya lebih bagus dari filmnya sendiri",
    "Naskah yang lemah dan karakter yang tidak berkembang",
    "Saya menyesal menonton film ini, buang-buang waktu",
]

# Gabungkan semua ulasan
semua_ulasan = ulasan_positif + ulasan_negatif
semua_label  = [1] * len(ulasan_positif) + [0] * len(ulasan_negatif)

# Buat sebagai DataFrame (tabel data)
df = pd.DataFrame({
    'teks' : semua_ulasan,
    'label': semua_label,
    'kelas': ['Positif' if l == 1 else 'Negatif' for l in semua_label]
})

# Acak urutan data (penting agar tidak bias)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print('✅ Dataset berhasil dibuat!')
print(f'   Total data  : {len(df)} ulasan')
print(f'   Ulasan positif (1): {sum(semua_label)}')
print(f'   Ulasan negatif (0): {len(semua_label) - sum(semua_label)}')
print()
print('5 contoh data pertama:')
df.head()

In [ ]:
# Simpan dataset ke file CSV
# CSV = Comma Separated Values, format tabel paling universal

df.to_csv('ulasan_film.csv', index=False)
print('✅ Dataset disimpan ke: ulasan_film.csv')
print()

# Verifikasi dengan membaca ulang
df_baca_ulang = pd.read_csv('ulasan_film.csv')
print(f'Verifikasi: berhasil dibaca kembali, {len(df_baca_ulang)} baris')
print()
print('Distribusi label:')
print(df['kelas'].value_counts())

---
## 🤗 LANGKAH 2: Upload Dataset ke HuggingFace

### Apa itu HuggingFace?
HuggingFace adalah platform seperti GitHub, tapi khusus untuk:
- **Dataset**: kumpulan data untuk training model
- **Model**: model AI yang sudah dilatih

Dengan upload ke HuggingFace, siapa pun di dunia bisa menggunakan
dataset atau model kamu hanya dengan satu baris kode Python.

### Langkah untuk mendapat Token HuggingFace:
1. Daftar di [huggingface.co](https://huggingface.co)
2. Klik foto profil → Settings → Access Tokens
3. Klik `New token` → pilih `Write` → beri nama → Copy token

---

In [ ]:
# ============================================================
# MEMBAGI DATA: TRAINING / VALIDASI / TEST
# ============================================================
#
# Kenapa dibagi 3?
# - TRAIN : untuk melatih model (80%)
# - VALID : untuk cek model saat training, hindari overfitting (10%)
# - TEST  : pengujian akhir, TIDAK BOLEH dilihat model saat training (10%)
#
# Analogi:
# - Train = soal latihan
# - Valid = try-out
# - Test  = ujian yang sebenarnya

# Split pertama: 80% train, 20% sisanya
df_train, df_temp = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']   # pastikan proporsi kelas tetap sama
)

# Split kedua: 50/50 dari sisa (masing-masing 10%)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=42,
    stratify=df_temp['label']
)

print('✅ Data berhasil dibagi!')
print(f'   Training   : {len(df_train)} data')
print(f'   Validasi   : {len(df_val)} data')
print(f'   Test       : {len(df_test)} data')
print(f'   Total      : {len(df_train)+len(df_val)+len(df_test)} data')
print()

# Cek distribusi di setiap split
for nama, data in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos = data['label'].sum()
    neg = len(data) - pos
    print(f'   {nama:5}: Positif={pos}, Negatif={neg}')

In [ ]:
# ============================================================
# KONVERSI KE FORMAT HUGGINGFACE DATASET
# ============================================================

# Buat HuggingFace DatasetDict
# DatasetDict = kamus berisi beberapa Dataset

hf_dataset = DatasetDict({
    'train'     : HFDataset.from_pandas(df_train.reset_index(drop=True)),
    'validation': HFDataset.from_pandas(df_val.reset_index(drop=True)),
    'test'      : HFDataset.from_pandas(df_test.reset_index(drop=True)),
})

print('✅ Format HuggingFace Dataset berhasil dibuat!')
print()
print('Struktur dataset:')
print(hf_dataset)
print()
print('Contoh 1 data dari training set:')
print(hf_dataset['train'][0])

In [ ]:
# ============================================================
# UPLOAD DATASET KE HUGGINGFACE
# ============================================================

# Ganti dengan username dan token HuggingFace kamu
HF_USERNAME = 'username_kamu'    # ← ganti ini
HF_TOKEN    = 'hf_XXXXX'         # ← ganti dengan token asli

NAMA_DATASET = f'{HF_USERNAME}/ulasan-film-indonesia'

print('📋 Cara upload dataset ke HuggingFace:')
print()
print('Step 1: Login')
print('   from huggingface_hub import login')
print("   login(token='hf_token_kamu')")
print()
print('Step 2: Upload')
print(f"   hf_dataset.push_to_hub('{NAMA_DATASET}')")
print()
print('Step 3: Akses dari mana saja')
print('   from datasets import load_dataset')
print(f"   ds = load_dataset('{NAMA_DATASET}')")
print()

# === JALANKAN INI SETELAH MENGISI TOKEN ASLI ===
# from huggingface_hub import login
# login(token=HF_TOKEN)
# hf_dataset.push_to_hub(NAMA_DATASET)
# print(f'✅ Dataset live di: https://huggingface.co/datasets/{NAMA_DATASET}')

# Untuk sekarang, simpan lokal dulu
hf_dataset.save_to_disk('dataset_ulasan_film')
print('✅ Dataset disimpan lokal di folder: dataset_ulasan_film/')

---
## 🧠 LANGKAH 3: Membuat Model dari Scratch

### Apa itu Model?
Model adalah fungsi matematika yang belajar dari data.
- **Input**: teks ulasan → diubah ke angka
- **Proses**: lewati beberapa layer matematika
- **Output**: probabilitas positif/negatif

### Yang kita buat: Model Sederhana (MLP)
MLP = Multi-Layer Perceptron — model neural network paling dasar.

```
INPUT (angka-angka fitur)
    ↓
[Layer 1: 64 neuron] → ReLU
    ↓
[Layer 2: 32 neuron] → ReLU
    ↓
[Output: 1 neuron] → Sigmoid
    ↓
OUTPUT (probabilitas 0.0 - 1.0)
```

**Catatan:** Model ini pakai fitur numerik (bukan teks langsung).
Untuk teks, kita akan pakai BERT di Langkah 5.

---

In [ ]:
# ============================================================
# BUAT FITUR NUMERIK DARI TEKS
# (Model sederhana tidak bisa baca teks langsung)
# ============================================================

def ekstrak_fitur(teks):
    """
    Ubah teks menjadi angka-angka yang bisa dibaca model sederhana.
    
    Ini disebut 'feature engineering' - membuat fitur dari data mentah.
    """
    teks_lower = teks.lower()
    kata_kata  = teks_lower.split()
    
    # Daftar kata positif dan negatif
    kata_positif = [
        'bagus', 'baik', 'luar biasa', 'menarik', 'recommended',
        'terbaik', 'memuaskan', 'menghibur', 'indah', 'keren',
        'sangat', 'banget', 'wajib', 'suka'
    ]
    kata_negatif = [
        'buruk', 'jelek', 'membosankan', 'mengecewakan', 'terburuk',
        'tidak', 'hambar', 'kaku', 'klise', 'murahan',
        'buang', 'menyesal', 'kecewa', 'amatir'
    ]
    
    # Hitung fitur-fitur sederhana
    fitur = [
        len(teks),                            # Panjang teks
        len(kata_kata),                        # Jumlah kata
        sum(1 for k in kata_positif if k in teks_lower),  # Kata positif
        sum(1 for k in kata_negatif if k in teks_lower),  # Kata negatif
        teks.count('!'),                       # Tanda seru
        teks_lower.count('sangat'),            # Kata intensitas
        teks_lower.count('tidak'),             # Kata negasi
        1 if 'terbaik' in teks_lower else 0,   # Kata superlative positif
        1 if 'terburuk' in teks_lower else 0,  # Kata superlative negatif
        sum(1 for k in kata_positif if k in teks_lower) -
        sum(1 for k in kata_negatif if k in teks_lower),  # Skor sentimen bersih
    ]
    return fitur

# Terapkan ke semua data
NAMA_FITUR = [
    'panjang_teks', 'jumlah_kata', 'kata_positif', 'kata_negatif',
    'tanda_seru', 'kata_sangat', 'kata_tidak',
    'ada_terbaik', 'ada_terburuk', 'skor_sentimen'
]

for nama_split, data_split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    fitur_list = [ekstrak_fitur(t) for t in data_split['teks']]
    df_fitur   = pd.DataFrame(fitur_list, columns=NAMA_FITUR)
    df_fitur.to_csv(f'fitur_{nama_split}.csv', index=False)

print('✅ Fitur numerik berhasil diekstrak!')
print(f'   Jumlah fitur per teks: {len(NAMA_FITUR)}')
print()
print('Contoh fitur dari ulasan pertama:')
contoh = df_train['teks'].iloc[0]
print(f'   Teks  : "{contoh[:50]}..."')
print(f'   Fitur : {ekstrak_fitur(contoh)}')
print()
pd.DataFrame([ekstrak_fitur(contoh)], columns=NAMA_FITUR)

In [ ]:
# ============================================================
# DEFINISI ARSITEKTUR MODEL (dari Scratch menggunakan PyTorch)
# ============================================================

class ModelSentimen(nn.Module):
    """
    Model klasifikasi sentimen sederhana.
    
    nn.Module = kelas dasar untuk semua model PyTorch
    Wajib implementasikan 2 method:
    - __init__ : definisi layer
    - forward  : cara data mengalir
    """
    
    def __init__(self, ukuran_input=10, hidden1=64, hidden2=32):
        """
        Definisi layer-layer dalam model.
        
        Parameter:
        - ukuran_input: jumlah fitur input (10 fitur kita)
        - hidden1: jumlah neuron layer 1
        - hidden2: jumlah neuron layer 2
        """
        super(ModelSentimen, self).__init__()
        
        # === LAYER 1: Input → Hidden ===
        self.layer1   = nn.Linear(ukuran_input, hidden1)
        # nn.Linear = y = Wx + b (persamaan linear)
        self.aktivasi1 = nn.ReLU()
        # ReLU = max(0, x) - buang nilai negatif
        self.dropout1  = nn.Dropout(0.3)
        # Dropout = matikan 30% neuron secara acak (cegah overfitting)
        
        # === LAYER 2: Hidden → Hidden ===
        self.layer2   = nn.Linear(hidden1, hidden2)
        self.aktivasi2 = nn.ReLU()
        self.dropout2  = nn.Dropout(0.2)
        
        # === LAYER OUTPUT: Hidden → 1 nilai ===
        self.output  = nn.Linear(hidden2, 1)
        self.sigmoid = nn.Sigmoid()
        # Sigmoid = konversi ke probabilitas 0.0–1.0
    
    def forward(self, x):
        """
        Cara data mengalir dari input ke output.
        Ini disebut 'forward pass'.
        """
        # Data mengalir: Input → Layer1 → Aktivasi → Dropout
        x = self.dropout1(self.aktivasi1(self.layer1(x)))
        # → Layer2 → Aktivasi → Dropout
        x = self.dropout2(self.aktivasi2(self.layer2(x)))
        # → Output → Sigmoid
        x = self.sigmoid(self.output(x))
        return x


# Buat instance model
model = ModelSentimen(ukuran_input=len(NAMA_FITUR)).to(DEVICE)

print('✅ Model berhasil dibuat!')
print()
print('Struktur model:')
print(model)
print()

# Hitung total parameter
total_param = sum(p.numel() for p in model.parameters())
print(f'Total parameter model : {total_param:,}')
print(f'(Parameter = angka-angka yang akan dipelajari model)')
print(f'(BERT punya 110 JUTA parameter - model ini sangat kecil!)')

---
## 🔥 LANGKAH 4: Melatih Model (Training)

### Proses Training (diulang ribuan kali):

```
1. FORWARD  : masukkan data ke model → dapat prediksi
2. LOSS     : hitung seberapa salah prediksi (error)
3. BACKWARD : hitung gradient (arah perbaikan)
4. UPDATE   : perbaiki bobot model
              ↑ ini disebut BACKPROPAGATION
```

### Istilah Penting:
| Istilah | Artinya |
|---------|--------|
| **Epoch** | Satu putaran training semua data |
| **Batch** | Kelompok data yang diproses sekaligus |
| **Loss** | Ukuran seberapa salah model (makin kecil makin baik) |
| **Learning Rate** | Seberapa besar langkah perbaikan |
| **Optimizer** | Algoritma pembaruan bobot (kita pakai Adam) |

---

In [ ]:
# ============================================================
# SIAPKAN DATA UNTUK TRAINING
# ============================================================

class DatasetTabular(Dataset):
    """
    Kelas Dataset untuk PyTorch.
    PyTorch butuh format khusus untuk membaca data secara efisien.
    """
    def __init__(self, df_data):
        fitur_list = [ekstrak_fitur(t) for t in df_data['teks']]
        self.X = torch.FloatTensor(fitur_list).to(DEVICE)
        self.y = torch.FloatTensor(df_data['label'].values).to(DEVICE)
    
    def __len__(self):
        """Berapa banyak data?"""
        return len(self.y)
    
    def __getitem__(self, idx):
        """Ambil data ke-idx"""
        return self.X[idx], self.y[idx]


# Normalisasi fitur (Z-score normalization)
# Mengapa perlu? Agar semua fitur punya skala yang sama
# Fitur panjang teks (0-200) vs kata positif (0-5) - beda skala!

fitur_train = np.array([ekstrak_fitur(t) for t in df_train['teks']])
mean_f = fitur_train.mean(axis=0)
std_f  = fitur_train.std(axis=0) + 1e-8  # +kecil agar tidak bagi 0

def df_norm(df_in):
    """Terapkan normalisasi ke DataFrame"""
    df_copy = df_in.copy()
    fitur   = np.array([ekstrak_fitur(t) for t in df_copy['teks']])
    fitur_n = (fitur - mean_f) / std_f
    df_copy2 = df_copy.copy()
    df_copy2['teks_asli'] = df_copy['teks']
    return df_copy2, fitur_n

# Buat dataset PyTorch
# (Untuk kemudahan, kita gunakan fitur yang sudah dinormalisasi)

class DatasetNorm(Dataset):
    def __init__(self, fitur_array, label_array):
        self.X = torch.FloatTensor(fitur_array).to(DEVICE)
        self.y = torch.FloatTensor(label_array).to(DEVICE)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

_, X_train = df_norm(df_train); y_train = df_train['label'].values
_, X_val   = df_norm(df_val);   y_val   = df_val['label'].values
_, X_test  = df_norm(df_test);  y_test  = df_test['label'].values

# Normalisasi manual pakai mean/std training
def norm_arr(arr): return (arr - mean_f) / std_f
X_train_n = norm_arr(X_train)
X_val_n   = norm_arr(X_val)
X_test_n  = norm_arr(X_test)

ds_train = DatasetNorm(X_train_n, y_train)
ds_val   = DatasetNorm(X_val_n,   y_val)
ds_test  = DatasetNorm(X_test_n,  y_test)

# DataLoader = alat yang menyajikan data dalam batch
loader_train = DataLoader(ds_train, batch_size=8, shuffle=True)
loader_val   = DataLoader(ds_val,   batch_size=8, shuffle=False)
loader_test  = DataLoader(ds_test,  batch_size=8, shuffle=False)

print('✅ Data siap untuk training!')
print(f'   Batch size: 8 (proses 8 data sekaligus)')
print(f'   Jumlah batch training: {len(loader_train)}')

In [ ]:
# ============================================================
# FUNGSI TRAINING DAN EVALUASI
# ============================================================

def latih_satu_epoch(model, loader, loss_fn, optimizer):
    """
    Melatih model selama satu epoch.
    
    Satu epoch = model melihat SEMUA data training satu kali.
    """
    model.train()   # Mode training: aktifkan dropout
    total_loss = 0
    
    for batch_X, batch_y in loader:
        # LANGKAH 1: Nol-kan gradient dari batch sebelumnya
        optimizer.zero_grad()
        
        # LANGKAH 2: Forward pass - model membuat prediksi
        prediksi = model(batch_X).squeeze()
        
        # LANGKAH 3: Hitung loss (seberapa salah prediksi)
        loss = loss_fn(prediksi, batch_y)
        
        # LANGKAH 4: Backward pass - hitung gradient
        # Gradient = arah dan besarnya perubahan yang dibutuhkan
        loss.backward()
        
        # LANGKAH 5: Update bobot model
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluasi(model, loader, loss_fn, threshold=0.5):
    """
    Evaluasi model pada data val/test.
    
    threshold: nilai probabilitas minimum untuk diklasifikasikan positif
    """
    model.eval()    # Mode evaluasi: matikan dropout
    total_loss = 0
    semua_pred = []
    semua_prob = []
    semua_label = []
    
    # torch.no_grad(): tidak perlu hitung gradient saat evaluasi
    # (lebih hemat memori dan lebih cepat)
    with torch.no_grad():
        for batch_X, batch_y in loader:
            prob = model(batch_X).squeeze()
            loss = loss_fn(prob, batch_y)
            total_loss += loss.item()
            
            # Konversi probabilitas ke label (0 atau 1)
            pred = (prob > threshold).float()
            
            semua_pred.extend(pred.cpu().numpy())
            semua_prob.extend(prob.cpu().numpy())
            semua_label.extend(batch_y.cpu().numpy())
    
    rata_loss = total_loss / len(loader)
    akurasi   = accuracy_score(semua_label, semua_pred)
    f1        = f1_score(semua_label, semua_pred, zero_division=0)
    
    return rata_loss, akurasi, f1, semua_pred, semua_prob, semua_label


print('✅ Fungsi training dan evaluasi siap!')

In [ ]:
# ============================================================
# TRAINING LOOP UTAMA
# ============================================================

# Setup komponen training
loss_fn   = nn.BCELoss()    # Binary Cross Entropy - untuk klasifikasi biner
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Adam = optimizer paling populer, adaptif learning rate

TOTAL_EPOCH = 50
best_f1     = 0
best_bobot  = None
riwayat     = []

print('🚀 Mulai training...')
print()
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Loss":>9} | {"Val Acc":>8} | {"Val F1":>7} |')
print('─' * 58)

for epoch in range(1, TOTAL_EPOCH + 1):
    
    # Training 1 epoch
    train_loss = latih_satu_epoch(model, loader_train, loss_fn, optimizer)
    
    # Evaluasi pada validation set
    val_loss, val_acc, val_f1, _, _, _ = evaluasi(model, loader_val, loss_fn)
    
    # Simpan riwayat
    riwayat.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1})
    
    # Simpan model terbaik berdasarkan val_f1
    # Ini disebut 'early stopping' - ambil yang terbaik
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_bobot = {k: v.clone() for k, v in model.state_dict().items()}
    
    # Tampilkan setiap 10 epoch
    if epoch % 10 == 0 or epoch == 1:
        tanda = ' ← best!' if val_f1 == best_f1 else ''
        print(f'{epoch:>6} | {train_loss:>10.4f} | {val_loss:>9.4f} | '
              f'{val_acc:>8.3f} | {val_f1:>7.3f} |{tanda}')

# Load model terbaik
model.load_state_dict(best_bobot)

print()
print(f'✅ Training selesai!')
print(f'   Model terbaik: Val F1 = {best_f1:.3f}')

---
## 📏 LANGKAH 6A: Evaluasi Model Sederhana

### Metrik Evaluasi yang Digunakan:

| Metrik | Artinya | Rumus |
|--------|---------|-------|
| **Accuracy** | Berapa % prediksi yang benar | TP+TN / Total |
| **Precision** | Dari yang diprediksi positif, berapa % benar? | TP / (TP+FP) |
| **Recall** | Dari yang benar positif, berapa % tertangkap? | TP / (TP+FN) |
| **F1-Score** | Rata-rata harmonis Precision dan Recall | 2×P×R/(P+R) |
| **AUC-ROC** | Area under ROC curve (0.5=random, 1.0=sempurna) | - |

### Confusion Matrix:
```
                PREDIKSI
              Negatif  Positif
AKTUAL  Neg |   TN   |   FP   |  ← FP = False Positive (salah positif)
        Pos |   FN   |   TP   |  ← FN = False Negative (terlewat!)
```

---

In [ ]:
# ============================================================
# EVALUASI LENGKAP PADA TEST SET
# ============================================================

test_loss, test_acc, test_f1, pred_test, prob_test, label_test = \
    evaluasi(model, loader_test, loss_fn)

# Hitung AUC
test_auc = roc_auc_score(label_test, prob_test)

# Confusion matrix
cm = confusion_matrix(label_test, pred_test)

print('=' * 55)
print('       HASIL EVALUASI MODEL SEDERHANA (MLP)')
print('       Tugas: Klasifikasi Sentimen Ulasan Film')
print('=' * 55)
print(f'Accuracy  : {test_acc:.4f}  ({test_acc*100:.1f}%)')
print(f'F1-Score  : {test_f1:.4f}')
print(f'ROC-AUC   : {test_auc:.4f}')
print(f'Test Loss : {test_loss:.4f}')
print()
print('Confusion Matrix:')
print('         Pred: Negatif  Positif')
print(f'Aktual: Neg     {cm[0,0]:>3}      {cm[0,1]:>3}')
print(f'Aktual: Pos     {cm[1,0]:>3}      {cm[1,1]:>3}')
print()
print('Detail per kelas:')
print(classification_report(label_test, pred_test,
      target_names=['Negatif', 'Positif']))

# Simpan hasil evaluasi
HASIL_MLP = {'acc': test_acc, 'f1': test_f1, 'auc': test_auc}

In [ ]:
# ============================================================
# COBA PREDIKSI MANUAL
# ============================================================

def prediksi_sentimen(teks, model=model, threshold=0.5):
    """
    Prediksi sentimen untuk sebuah teks baru.
    """
    model.eval()
    fitur    = ekstrak_fitur(teks)
    fitur_n  = (np.array(fitur) - mean_f) / std_f
    tensor_x = torch.FloatTensor(fitur_n).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        prob = model(tensor_x).item()
    
    kelas = 'POSITIF 😊' if prob > threshold else 'NEGATIF 😞'
    return kelas, prob


# Test dengan beberapa contoh
contoh_baru = [
    "Film yang sangat bagus dan sangat menghibur!",
    "Sangat membosankan dan tidak ada yang menarik",
    "Ceritanya biasa saja, tidak bagus tidak buruk",
    "Luar biasa! Film terbaik yang pernah saya tonton!",
    "Terburuk, buang waktu saja menonton ini",
]

print('🎬 Prediksi Sentimen Ulasan Film Baru:')
print('─' * 65)
for teks in contoh_baru:
    kelas, prob = prediksi_sentimen(teks)
    print(f'Ulasan  : "{teks[:50]}"')
    print(f'Prediksi: {kelas} (probabilitas: {prob:.3f})')
    print()

---
## 🤖 LANGKAH 5: Fine-tuning BERT

### Apa itu BERT?
BERT (Bidirectional Encoder Representations from Transformers) adalah
model bahasa yang sudah dilatih Google pada **miliaran teks** dari internet.

BERT sudah 'paham' bahasa secara mendalam. Kita tinggal **fine-tune**
(melatih ulang sedikit) untuk task spesifik kita.

### Analogi:
```
BERT = Chef berpengalaman yang sudah ahli masak
                    ↓ fine-tuning
Chef yang sama, tapi sekarang spesialis masakan Padang
```

### Strategi Efisien (Frozen Layers):
```
Layer 1-10  : FREEZE  (tidak diubah - sudah tahu bahasa)
Layer 11-12 : TRAIN   (disesuaikan untuk task kita)
Classifier  : TRAIN   (layer baru untuk klasifikasi)
```

---

In [ ]:
# ============================================================
# PERSIAPAN FINE-TUNING BERT
# ============================================================

print('📥 Mendownload BERT tokenizer...')
print('(Pertama kali: download ~500KB, selanjutnya dari cache)')
print()

# Tokenizer = alat yang mengubah teks ke token ID
# bert-base-uncased = BERT versi lowercase (semua huruf kecil)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Demonstrasi tokenisasi
contoh = "Film ini sangat bagus!"
hasil  = tokenizer(contoh)

print('Contoh tokenisasi:')
print(f'  Teks asli  : "{contoh}"')
print(f'  Token IDs  : {hasil["input_ids"]}')
print(f'  Token text : {tokenizer.convert_ids_to_tokens(hasil["input_ids"])}')
print()
print('Penjelasan:')
print('  [CLS] = token awal (tanda mulai)')
print('  [SEP] = token akhir (tanda selesai)')
print('  ##ini = bagian dari kata sebelumnya (sub-word)')

In [ ]:
# ============================================================
# DATASET UNTUK BERT
# ============================================================

class DatasetBERT(Dataset):
    """
    Dataset khusus untuk fine-tuning BERT.
    
    BERT butuh 3 input:
    1. input_ids      : token ID teks
    2. attention_mask : mana token asli (1) vs padding (0)
    3. labels         : label yang benar (0 atau 1)
    """
    def __init__(self, df_data, tokenizer, max_panjang=64):
        self.tokenizer  = tokenizer
        self.teks       = df_data['teks'].tolist()
        self.label      = df_data['label'].tolist()
        self.max_panjang = max_panjang
    
    def __len__(self):
        return len(self.label)
    
    def __getitem__(self, idx):
        # Tokenisasi satu teks
        encoding = self.tokenizer(
            self.teks[idx],
            max_length=self.max_panjang,
            padding='max_length',    # padding ke panjang maksimal
            truncation=True,          # potong jika terlalu panjang
            return_tensors='pt'       # return PyTorch tensor
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.LongTensor([self.label[idx]])
        }


# Buat dataset
bert_ds_train = DatasetBERT(df_train, tokenizer)
bert_ds_val   = DatasetBERT(df_val,   tokenizer)
bert_ds_test  = DatasetBERT(df_test,  tokenizer)

# DataLoader
bert_loader_train = DataLoader(bert_ds_train, batch_size=8, shuffle=True)
bert_loader_val   = DataLoader(bert_ds_val,   batch_size=8, shuffle=False)
bert_loader_test  = DataLoader(bert_ds_test,  batch_size=8, shuffle=False)

print('✅ Dataset BERT berhasil dibuat!')
print()

# Lihat satu contoh
contoh_item = bert_ds_train[0]
print('Contoh 1 item dari dataset BERT:')
print(f'  input_ids shape     : {contoh_item["input_ids"].shape}')
print(f'  attention_mask shape: {contoh_item["attention_mask"].shape}')
print(f'  label               : {contoh_item["labels"]}')

In [ ]:
# ============================================================
# LOAD BERT DAN BUAT MODEL FINE-TUNING
# ============================================================

print('📥 Mendownload BERT model...')
print('(Pertama kali: download ~440MB - mungkin 2-5 menit)')
print()

# BertForSequenceClassification = BERT + layer klasifikasi
# num_labels=2 karena kita punya 2 kelas: Negatif dan Positif
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
).to(DEVICE)

# STRATEGI FREEZE:
# Freeze layer 0-9, hanya latih layer 10, 11, dan classifier
for nama, param in bert_model.bert.named_parameters():
    # Freeze semua kecuali 2 layer terakhir
    if ('encoder.layer.10' not in nama and
        'encoder.layer.11' not in nama and
        'pooler' not in nama):
        param.requires_grad = False

# Hitung statistik parameter
total_param    = sum(p.numel() for p in bert_model.parameters())
param_dilatih  = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
param_difreeze = total_param - param_dilatih

print('✅ BERT berhasil diload!')
print()
print('Statistik parameter:')
print(f'  Total parameter      : {total_param:>12,}')
print(f'  Yang akan dilatih    : {param_dilatih:>12,} ({param_dilatih/total_param*100:.1f}%)')
print(f'  Yang di-freeze       : {param_difreeze:>12,} ({param_difreeze/total_param*100:.1f}%)')
print()
print('Dengan freeze, kita hanya melatih bagian kecil BERT → lebih cepat!')

In [ ]:
# ============================================================
# FINE-TUNING BERT
# ============================================================

bert_optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=2e-5,      # Learning rate kecil untuk fine-tuning
    eps=1e-8      # Epsilon untuk stabilitas
)

BERT_EPOCH = 3  # BERT tidak butuh banyak epoch!
best_bert_f1    = 0
best_bert_bobot = None

print('🚀 Mulai fine-tuning BERT...')
print(f'   Hanya {BERT_EPOCH} epoch (BERT sudah sangat pintar!)')
print()
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Acc":>8} | {"Val F1":>7} |')
print('─' * 45)

for epoch in range(1, BERT_EPOCH + 1):
    
    # Training
    bert_model.train()
    total_loss = 0
    
    for batch in bert_loader_train:
        bert_optimizer.zero_grad()
        
        # BERT forward pass - langsung return loss jika labels diberikan
        output = bert_model(
            input_ids      = batch['input_ids'].to(DEVICE),
            attention_mask = batch['attention_mask'].to(DEVICE),
            labels         = batch['labels'].squeeze().to(DEVICE)
        )
        
        loss = output.loss
        loss.backward()
        
        # Clip gradient - cegah gradient exploding
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        
        bert_optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(bert_loader_train)
    
    # Evaluasi
    bert_model.eval()
    pred_val, label_val = [], []
    
    with torch.no_grad():
        for batch in bert_loader_val:
            output = bert_model(
                input_ids      = batch['input_ids'].to(DEVICE),
                attention_mask = batch['attention_mask'].to(DEVICE)
            )
            pred = torch.argmax(output.logits, dim=1)
            pred_val.extend(pred.cpu().numpy())
            label_val.extend(batch['labels'].squeeze().numpy())
    
    v_acc = accuracy_score(label_val, pred_val)
    v_f1  = f1_score(label_val, pred_val, zero_division=0)
    
    if v_f1 > best_bert_f1:
        best_bert_f1    = v_f1
        best_bert_bobot = {k: v.clone() for k, v in bert_model.state_dict().items()}
    
    tanda = ' ← best!' if v_f1 == best_bert_f1 else ''
    print(f'{epoch:>6} | {avg_loss:>10.4f} | {v_acc:>8.3f} | {v_f1:>7.3f} |{tanda}')

bert_model.load_state_dict(best_bert_bobot)
print()
print(f'✅ Fine-tuning selesai! Best Val F1: {best_bert_f1:.3f}')

In [ ]:
# ============================================================
# EVALUASI BERT PADA TEST SET
# ============================================================

bert_model.eval()
pred_test_bert = []
prob_test_bert = []
label_test_bert = []

with torch.no_grad():
    for batch in bert_loader_test:
        output = bert_model(
            input_ids      = batch['input_ids'].to(DEVICE),
            attention_mask = batch['attention_mask'].to(DEVICE)
        )
        prob = torch.softmax(output.logits, dim=1)[:, 1]
        pred = torch.argmax(output.logits, dim=1)
        prob_test_bert.extend(prob.cpu().numpy())
        pred_test_bert.extend(pred.cpu().numpy())
        label_test_bert.extend(batch['labels'].squeeze().numpy())

bert_acc = accuracy_score(label_test_bert, pred_test_bert)
bert_f1  = f1_score(label_test_bert, pred_test_bert, zero_division=0)
bert_auc = roc_auc_score(label_test_bert, prob_test_bert)

print('=' * 55)
print('    HASIL EVALUASI BERT (Fine-tuned)')
print('    Tugas: Klasifikasi Sentimen Ulasan Film')
print('=' * 55)
print(f'Accuracy : {bert_acc:.4f}  ({bert_acc*100:.1f}%)')
print(f'F1-Score : {bert_f1:.4f}')
print(f'AUC-ROC  : {bert_auc:.4f}')
print()
print(classification_report(label_test_bert, pred_test_bert,
      target_names=['Negatif', 'Positif']))

HASIL_BERT = {'acc': bert_acc, 'f1': bert_f1, 'auc': bert_auc}

print('=== PERBANDINGAN MODEL ===')
print(f'            MLP (tabular)    BERT (teks)')
print(f'Accuracy :  {HASIL_MLP["acc"]:.4f}          {HASIL_BERT["acc"]:.4f}')
print(f'F1-Score :  {HASIL_MLP["f1"]:.4f}          {HASIL_BERT["f1"]:.4f}')
print(f'AUC-ROC  :  {HASIL_MLP["auc"]:.4f}          {HASIL_BERT["auc"]:.4f}')
print()
print('💡 BERT menggunakan teks langsung → biasanya lebih akurat untuk NLP task')

In [ ]:
# Coba prediksi manual dengan BERT
def prediksi_bert(teks, model=bert_model, tokenizer=tokenizer):
    """Prediksi sentimen menggunakan BERT"""
    model.eval()
    enc = tokenizer(teks, return_tensors='pt',
                   padding=True, truncation=True, max_length=64)
    with torch.no_grad():
        out  = model(input_ids=enc['input_ids'].to(DEVICE),
                     attention_mask=enc['attention_mask'].to(DEVICE))
        prob = torch.softmax(out.logits, dim=1)[0]
        pred = torch.argmax(prob).item()
    kelas = 'POSITIF 😊' if pred == 1 else 'NEGATIF 😞'
    return kelas, prob[1].item()

print('🎬 Prediksi BERT untuk ulasan baru:')
print('─' * 65)
for teks in contoh_baru:
    kelas, prob = prediksi_bert(teks)
    print(f'Ulasan  : "{teks[:55]}"')
    print(f'BERT    : {kelas} (prob positif: {prob:.3f})')
    print()

---
## 💾 LANGKAH 7: Simpan dan Upload Model ke HuggingFace

### Mengapa Upload ke HuggingFace?
1. **Reproducibility**: Peneliti lain bisa verifikasi hasilmu
2. **Citability**: Model mendapat DOI lewat Zenodo
3. **Accessibility**: Load model dengan 1 baris kode
4. **Standar ilmiah**: Semakin banyak jurnal Q1 mensyaratkan ini

---

In [ ]:
# ============================================================
# SIMPAN MODEL KE DISK
# ============================================================

import os, json

FOLDER_MODEL = './sentiment-bert'
os.makedirs(FOLDER_MODEL, exist_ok=True)

# Simpan model BERT (weights + konfigurasi)
bert_model.save_pretrained(FOLDER_MODEL)

# Simpan tokenizer
tokenizer.save_pretrained(FOLDER_MODEL)

# Simpan hasil evaluasi
with open(f'{FOLDER_MODEL}/eval_results.json', 'w') as f:
    json.dump({'mlp': HASIL_MLP, 'bert': HASIL_BERT}, f, indent=2)

print(f'✅ Model berhasil disimpan di: {FOLDER_MODEL}/')
print()
print('File yang tersimpan:')
for f in sorted(os.listdir(FOLDER_MODEL)):
    ukuran = os.path.getsize(f'{FOLDER_MODEL}/{f}') / 1e6
    print(f'  {f:<30} {ukuran:.1f} MB')

In [ ]:
# ============================================================
# BUAT MODEL CARD (README.md)
# Model Card = identitas dan dokumentasi model
# ============================================================

model_card_isi = f"""---
language: id
license: mit
tags:
  - text-classification
  - sentiment-analysis
  - bert
  - indonesian
datasets:
  - username_kamu/ulasan-film-indonesia
metrics:
  - accuracy
  - f1
  - roc_auc
---

# Sentiment BERT - Klasifikasi Sentimen Ulasan Film

## Deskripsi
Model BERT yang di-fine-tune untuk klasifikasi sentimen 
ulasan film berbahasa Indonesia.

- **Label**: 0 = Negatif, 1 = Positif
- **Base model**: bert-base-uncased
- **Task**: Sequence Classification (Binary)

## Cara Pakai
```python
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained('username_kamu/sentiment-bert')
model = BertForSequenceClassification.from_pretrained('username_kamu/sentiment-bert')

teks = "Film ini sangat bagus dan menghibur!"
inputs = tokenizer(teks, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()

print('Positif' if pred == 1 else 'Negatif')
```

## Hasil Evaluasi
| Metrik   | MLP Tabular | BERT Fine-tuned |
|----------|-------------|----------------|
| Accuracy | {HASIL_MLP['acc']:.4f}      | {HASIL_BERT['acc']:.4f}         |
| F1-Score | {HASIL_MLP['f1']:.4f}      | {HASIL_BERT['f1']:.4f}         |
| AUC-ROC  | {HASIL_MLP['auc']:.4f}      | {HASIL_BERT['auc']:.4f}         |

## Cara Replikasi
1. Download tutorial notebook dari repository
2. Jalankan di Google Colab
3. Semua langkah tersedia dan terdokumentasi

## Keterbatasan
- Data training sangat kecil (30 contoh) - hanya untuk demonstrasi
- Untuk produksi, gunakan data ribuan atau jutaan contoh
"""

with open(f'{FOLDER_MODEL}/README.md', 'w', encoding='utf-8') as f:
    f.write(model_card_isi)

print('✅ Model Card (README.md) berhasil dibuat!')
print()
print('Isi Model Card:')
print(model_card_isi[:500] + '...')

In [ ]:
# ============================================================
# UPLOAD MODEL KE HUGGINGFACE
# ============================================================

NAMA_MODEL = f'{HF_USERNAME}/sentiment-bert'

print('📋 Langkah upload model ke HuggingFace:')
print()
print('1. Pastikan sudah login:')
print('   from huggingface_hub import login')
print("   login(token='hf_token_kamu')")
print()
print('2. Upload model:')
print(f"   bert_model.push_to_hub('{NAMA_MODEL}')")
print()
print('3. Upload tokenizer:')
print(f"   tokenizer.push_to_hub('{NAMA_MODEL}')")
print()
print('4. Setelah upload, siapa pun bisa pakai dengan:')
print(f"   from transformers import pipeline")
print(f"   nlp = pipeline('text-classification', model='{NAMA_MODEL}')")
print(f"   hasil = nlp('Film ini bagus sekali!')")
print(f"   print(hasil)")
print()

# === JALANKAN INI SETELAH MENGISI HF_TOKEN ASLI ===
# from huggingface_hub import login
# login(token=HF_TOKEN)
# bert_model.push_to_hub(NAMA_MODEL)
# tokenizer.push_to_hub(NAMA_MODEL)
# hf_dataset.push_to_hub(f'{HF_USERNAME}/ulasan-film-indonesia')
# print(f'✅ Model live di: https://huggingface.co/{NAMA_MODEL}')

print('─' * 55)
print('✅ TUTORIAL SELESAI! Semua langkah berhasil dipelajari.')

---
## 🎓 Ringkasan Akhir

### Yang Sudah Kamu Pelajari:

```
📊 DATA
├── Membuat dataset (teks + label)
├── Menyimpan ke CSV
├── Membagi train/val/test
└── Upload ke HuggingFace

🧠 MODEL SEDERHANA (MLP)
├── Definisi arsitektur (nn.Module)
├── Feature engineering (teks → angka)
├── Training loop (forward → loss → backward → update)
├── Evaluasi (accuracy, F1, AUC, confusion matrix)
└── Prediksi data baru

🤖 BERT FINE-TUNING
├── Tokenisasi teks
├── Load pre-trained model
├── Freeze layer strategi
├── Fine-tuning loop
└── Evaluasi dan perbandingan

🤗 HUGGINGFACE
├── Upload dataset
├── Upload model + tokenizer
└── Model Card (dokumentasi)
```

### Konsep Kunci yang Harus Diingat:

| Konsep | Penjelasan Sederhana |
|--------|---------------------|
| **Epoch** | Satu putaran melihat semua data |
| **Batch** | Kelompok data yang diproses sekaligus |
| **Loss** | Ukuran kesalahan (makin kecil makin baik) |
| **Gradient** | Arah dan besar perbaikan yang perlu dilakukan |
| **Fine-tuning** | Melatih model yang sudah ada untuk task baru |
| **Overfitting** | Model hafal data training tapi buruk di data baru |
| **Dropout** | Teknik mencegah overfitting |
| **Seed** | Angka acak tetap agar hasil selalu sama |

---

### Langkah Belajar Selanjutnya:

1. **Ganti dataset**: gunakan data nyata yang lebih besar
2. **Coba model lain**: RoBERTa, DistilBERT, ALBERT
3. **Multi-class**: lebih dari 2 kelas
4. **Pelajari SHAP**: jelaskan kenapa model membuat prediksi tersebut
5. **Pelajari XAI**: gunakan SHAP atau LIME untuk interpretasi model

---

*Tutorial ini dibuat sebagai panduan belajar ML untuk pemula.*
*Semua konsep berlaku untuk penelitian AI di bidang apapun.*